In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import random
import os
import warnings
import json
from datetime import datetime
import copy

warnings.filterwarnings("ignore")
os.makedirs("./outputs", exist_ok=True)

# =========================
# CONFIG (EUROSAT)
# =========================
QUICK_RUN = True
SAVE_PATH = "./outputs/kfn_eurosat_progress.json"

# EuroSAT classes = 10 total
TOTAL_CLASSES = 10
OLD_CLASSES = 5
NEW_CLASSES = TOTAL_CLASSES - OLD_CLASSES

if QUICK_RUN:
    EPOCHS_A = 10
    EPOCHS_B = 20
    EPOCHS_BIAS = 6
    SEEDS = [0]
else:
    EPOCHS_A = 40
    EPOCHS_B = 120
    EPOCHS_BIAS = 12
    SEEDS = [0, 1, 2, 3, 4]

BS = 128
NW = 2
REPLAY_PER_CLASS = 500
EVAL_TEMP = 1.2

SUPCON_W_A = 0.12
SUPCON_W_B = 0.08
EMA_DECAY = 0.999

USE_AMP = True
FORCE_CPU = False

# =========================
# UTILS
# =========================
def set_deterministic(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device(force_cpu=False):
    if force_cpu:
        print("⚠️ FORCE_CPU=True -> using CPU")
        return torch.device("cpu")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def save_progress(obj, path=SAVE_PATH):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def build_class_index_map_from_indices(targets, indices):
    class_to_indices = {}
    for i in indices:
        y = targets[i]
        class_to_indices.setdefault(y, []).append(i)
    return class_to_indices

def sample_replay_indices(class_to_indices, classes, per_class, rng):
    indices = []
    for c in classes:
        pool = class_to_indices[c]
        if len(pool) <= per_class:
            indices.extend(pool)
        else:
            indices.extend(rng.sample(pool, per_class))
    rng.shuffle(indices)
    return indices

def make_dynamic_replay_loader(base_ds, class_to_indices, old_classes, per_class, bs, nw, rng):
    indices = sample_replay_indices(class_to_indices, old_classes, per_class, rng)
    subset = Subset(base_ds, indices)
    return DataLoader(subset, batch_size=bs, shuffle=True, num_workers=nw, pin_memory=True)

def update_ema(ema_model, model, decay=0.999):
    with torch.no_grad():
        msd = model.state_dict()
        for k, v in ema_model.state_dict().items():
            if v.dtype.is_floating_point:
                v.copy_(v * decay + msd[k] * (1.0 - decay))
            else:
                v.copy_(msd[k])

def stratified_train_test_indices(targets, test_ratio=0.2, seed=42):
    rng = random.Random(seed)
    cls_to_idx = {}
    for i, y in enumerate(targets):
        cls_to_idx.setdefault(y, []).append(i)

    train_idx, test_idx = [], []
    for c, idxs in cls_to_idx.items():
        idxs = idxs.copy()
        rng.shuffle(idxs)
        n_test = max(1, int(len(idxs) * test_ratio))
        test_idx.extend(idxs[:n_test])
        train_idx.extend(idxs[n_test:])

    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return train_idx, test_idx

def filter_indices_by_class(indices, targets, classes):
    classes = set(classes)
    return [i for i in indices if targets[i] in classes]

# =========================
# SupCon Loss
# =========================
def supervised_contrastive_loss(features, labels, temperature=0.07):
    device = features.device
    bsz = features.shape[0]

    logits = torch.matmul(features, features.T) / temperature
    logits_max, _ = torch.max(logits, dim=1, keepdim=True)
    logits = logits - logits_max.detach()

    labels = labels.contiguous().view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(device)

    logits_mask = torch.ones_like(mask) - torch.eye(bsz, device=device)
    mask = mask * logits_mask

    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)

    mask_sum = mask.sum(1)
    mean_log_prob_pos = (mask * log_prob).sum(1) / (mask_sum + 1e-12)

    loss = -mean_log_prob_pos
    valid = (mask_sum > 0).float()
    loss = (loss * valid).sum() / (valid.sum() + 1e-12)
    return loss

# =========================
# MODEL
# =========================
class CosineLinear(nn.Module):
    def __init__(self, in_features, out_features, sigma=30.0):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.sigma = nn.Parameter(torch.tensor([sigma], dtype=torch.float32))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))

    def forward(self, x):
        return self.sigma * F.linear(F.normalize(x, p=2, dim=1), F.normalize(self.weight, p=2, dim=1))

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)

class Specialist(nn.Module):
    """
    EuroSAT 64x64-friendly stronger backbone:
    ResNet-34 style depth: [3,4,6,3]
    """
    def __init__(self):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=5, stride=1, padding=2, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(BasicBlock, 64, 3, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 4, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 6, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 3, stride=2)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.layer1(x)   # 64x64
        x = self.layer2(x)   # 32x32
        x = self.layer3(x)   # 16x16
        x = self.layer4(x)   # 8x8
        return x

class StructuralFusionModule(nn.Module):
    def __init__(self, in_channels, old_dim, new_dim, novelty_score):
        super().__init__()
        self.novelty_score = novelty_score
        self.has_expansion = new_dim > 0

        self.proj_reuse = nn.Sequential(
            nn.Conv2d(in_channels, old_dim, 1, bias=False),
            nn.BatchNorm2d(old_dim), nn.ReLU(inplace=True),
            nn.Conv2d(old_dim, old_dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(old_dim), nn.ReLU(inplace=True),
            nn.Conv2d(old_dim, old_dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(old_dim), nn.ReLU(inplace=True),
        )
        self.gate_reuse = nn.Parameter(torch.tensor([0.0]))

        if self.has_expansion:
            self.proj_expand = nn.Sequential(
                nn.Conv2d(in_channels, new_dim, 1, bias=False),
                nn.BatchNorm2d(new_dim), nn.ReLU(inplace=True),
                nn.Conv2d(new_dim, new_dim, 3, padding=1, bias=False),
                nn.BatchNorm2d(new_dim), nn.ReLU(inplace=True),
                nn.Conv2d(new_dim, new_dim, 3, padding=1, bias=False),
                nn.BatchNorm2d(new_dim), nn.ReLU(inplace=True),
            )

    def forward(self, x, old_memory_detached):
        reuse_scale = max(0.1, 0.55 - self.novelty_score * 0.25)
        delta_old = self.proj_reuse(x) * torch.sigmoid(self.gate_reuse) * reuse_scale

        feat_new = None
        if self.has_expansion:
            raw_new = self.proj_expand(x)
            feat_new = raw_new - raw_new.mean(dim=(2, 3), keepdim=True)

        return delta_old, feat_new

class GlobalModel(nn.Module):
    def __init__(self, n_specs, ch, n_classes, old_classes, old_dim=256, new_dim=0, novelty_score=0.0, use_weight_align=True):
        super().__init__()
        self.old_dim = old_dim
        self.new_dim = new_dim
        self.old_classes = old_classes
        self.use_weight_align = use_weight_align

        self.old_proj = nn.ModuleList([nn.Conv2d(ch, ch, 1)])
        self.old_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, ch // 4, 1), nn.ReLU(inplace=True),
            nn.Conv2d(ch // 4, ch, 1), nn.Sigmoid()
        )
        self.old_bottleneck = nn.Sequential(nn.Conv2d(ch, old_dim, 1), nn.ReLU(inplace=True))

        self.fusion = StructuralFusionModule(ch, old_dim, new_dim, novelty_score) if new_dim > 0 else None
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = CosineLinear(old_dim + new_dim, n_classes, sigma=30.0)

    def forward_old(self, feats):
        p = self.old_proj[0](feats[0])
        z = p * self.old_gate(p)
        return self.old_bottleneck(z)

    def forward(self, feats):
        old_memory = self.forward_old(feats)

        if self.fusion is not None:
            spec_new = feats[-1]
            delta_old, feat_new = self.fusion(spec_new, old_memory.detach())
            enhanced_old = old_memory + delta_old
            final_features = torch.cat([enhanced_old, feat_new], dim=1) if feat_new is not None else enhanced_old
        else:
            final_features = old_memory

        split = self.old_classes

        W_old = self.classifier.weight[:split, :self.old_dim]
        W_new = self.classifier.weight[split:, :]

        flat_old = F.normalize(self.pool(old_memory).flatten(1), dim=1)
        logits_old = F.linear(flat_old, F.normalize(W_old, p=2, dim=1)) * self.classifier.sigma

        flat_new = F.normalize(self.pool(final_features).flatten(1), dim=1)
        logits_new = F.linear(flat_new, F.normalize(W_new, p=2, dim=1)) * self.classifier.sigma

        logits = torch.cat([logits_old, logits_new], dim=1)

        # safe WA only
        if self.use_weight_align and logits.size(1) >= TOTAL_CLASSES:
            with torch.no_grad():
                w_old = self.classifier.weight[:split]
                w_new = self.classifier.weight[split:]
                norm_old = w_old.norm(dim=1).mean()
                norm_new = w_new.norm(dim=1).mean()
                gamma = norm_old / (norm_new + 1e-8)
            logits[:, split:] *= gamma

        return logits, old_memory

class KFN(nn.Module):
    def __init__(self, n_classes=OLD_CLASSES, old_classes=OLD_CLASSES, n_specs=1, ch=512, old_dim=256, new_dim=0, novelty_score=0.0):
        super().__init__()
        self.specialists = nn.ModuleList([Specialist() for _ in range(n_specs)])
        self.global_model = GlobalModel(
            n_specs, ch, n_classes, old_classes=old_classes,
            old_dim=old_dim, new_dim=new_dim, novelty_score=novelty_score, use_weight_align=True
        )

    def forward(self, x):
        return self.global_model([s(x) for s in self.specialists])

# =========================
# TRAIN / EVAL
# =========================
def expand_kfn(old_model, trained_spec, new_dim, novelty_score, device):
    new_kfn = KFN(
        n_classes=TOTAL_CLASSES,
        old_classes=OLD_CLASSES,
        n_specs=2,
        ch=512,
        old_dim=256,
        new_dim=new_dim,
        novelty_score=novelty_score
    ).to(device)

    new_kfn.specialists[0].load_state_dict(old_model.specialists[0].state_dict())
    new_kfn.specialists[1].load_state_dict(trained_spec.state_dict())

    old_g = old_model.global_model
    new_g = new_kfn.global_model
    new_g.old_proj.load_state_dict(old_g.old_proj.state_dict())
    new_g.old_gate.load_state_dict(old_g.old_gate.state_dict())
    new_g.old_bottleneck.load_state_dict(old_g.old_bottleneck.state_dict())

    with torch.no_grad():
        new_g.classifier.weight[:OLD_CLASSES, :new_g.old_dim] = old_g.classifier.weight[:, :old_g.old_dim]
        new_g.classifier.sigma.data = old_g.classifier.sigma.data.clone()
        nn.init.kaiming_normal_(new_g.classifier.weight[OLD_CLASSES:, :], nonlinearity='relu')
        if new_g.new_dim > 0:
            new_g.classifier.weight[:OLD_CLASSES, new_g.old_dim:].zero_()

    return new_kfn

def compute_novelty(model, specialist, loader, device):
    model.eval()
    specialist.eval()
    scores = []

    gen = torch.Generator(device="cpu")
    gen.manual_seed(12345)
    proj_layer = nn.Conv2d(512, model.global_model.old_dim, 1, bias=False).to(device)
    with torch.no_grad():
        w = torch.empty_like(proj_layer.weight, device="cpu")
        w.normal_(generator=gen)
        proj_layer.weight.copy_(w.to(device))
    proj_layer.eval()

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            f_new = specialist(x)
            f_new = F.normalize(proj_layer(F.adaptive_avg_pool2d(f_new, 1)).flatten(1), p=2, dim=1)

            f_old = model.specialists[0](x)
            f_old = model.global_model.forward_old([f_old])
            f_old = F.normalize(F.adaptive_avg_pool2d(f_old, 1).flatten(1), p=2, dim=1)

            proj = torch.sum(f_new * f_old, dim=1, keepdim=True) * f_old
            residual = f_new - proj
            scores.append(torch.norm(residual, p=2, dim=1).mean().item())

    avg = float(np.mean(scores))
    new_dim = 768 if avg > 0.2 else 512
    return new_dim, avg

def evaluate(model, loader, device, mode="cil", temp=1.2):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            logits = logits / temp

            if mode == "task_A":
                pred = logits[:, :OLD_CLASSES].argmax(1)
            elif mode == "task_B":
                pred = logits[:, OLD_CLASSES:].argmax(1) + OLD_CLASSES
            else:
                pred = logits.argmax(1)

            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total

def run_kfn_eurosat(seed, loaders, epochs_A, epochs_B, epochs_bias, device, train_base, class_to_indices_train):
    set_deterministic(seed)
    train_A, train_B, test_A, test_B = loaders
    rng = random.Random(seed + 999)

    amp_ok = USE_AMP and (device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda") if amp_ok else None

    # -------- Phase 1 (Task A)
    model = KFN(n_classes=OLD_CLASSES, old_classes=OLD_CLASSES, n_specs=1, ch=512, old_dim=256).to(device)
    opt1 = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=epochs_A)

    for _ in range(epochs_A):
        model.train()
        for x, y in train_A:
            x, y = x.to(device), y.to(device)
            opt1.zero_grad()

            if amp_ok:
                with torch.amp.autocast("cuda"):
                    logits, _ = model(x)
                    ce = F.cross_entropy(logits, y, label_smoothing=0.1)
                    feat_map = model.specialists[0](x)
                    feat = F.normalize(F.adaptive_avg_pool2d(feat_map, 1).flatten(1), dim=1)
                    supcon = supervised_contrastive_loss(feat, y)
                    loss = ce + SUPCON_W_A * supcon
                scaler.scale(loss).backward()
                scaler.step(opt1)
                scaler.update()
            else:
                logits, _ = model(x)
                ce = F.cross_entropy(logits, y, label_smoothing=0.1)
                feat_map = model.specialists[0](x)
                feat = F.normalize(F.adaptive_avg_pool2d(feat_map, 1).flatten(1), dim=1)
                supcon = supervised_contrastive_loss(feat, y)
                loss = ce + SUPCON_W_A * supcon
                loss.backward()
                opt1.step()

        sched1.step()

    acc_A_init = evaluate(model, test_A, device, mode="task_A", temp=EVAL_TEMP)

    # -------- Phase 2 (new specialist on Task B)
    spec = Specialist().to(device)
    head = nn.Linear(512, NEW_CLASSES).to(device)
    epochs_spec = int(0.8 * epochs_B)

    opt2 = torch.optim.AdamW(list(spec.parameters()) + list(head.parameters()), lr=1e-3)
    sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs_spec)

    for ep in range(epochs_spec):
        spec.train()
        for x, y in train_B:
            x = x.to(device)
            y_raw = y.to(device)
            y = y_raw - OLD_CLASSES
            opt2.zero_grad()

            if amp_ok:
                with torch.amp.autocast("cuda"):
                    feat_map = spec(x)
                    feat = F.normalize(F.adaptive_avg_pool2d(feat_map, 1).flatten(1), dim=1)
                    logits = head(feat)
                    ce = F.cross_entropy(logits, y, label_smoothing=0.1)
                    if ep < epochs_spec // 2:
                        loss = ce
                    else:
                        supcon = supervised_contrastive_loss(feat, y)
                        loss = ce + SUPCON_W_B * supcon
                scaler.scale(loss).backward()
                scaler.step(opt2)
                scaler.update()
            else:
                feat_map = spec(x)
                feat = F.normalize(F.adaptive_avg_pool2d(feat_map, 1).flatten(1), dim=1)
                logits = head(feat)
                ce = F.cross_entropy(logits, y, label_smoothing=0.1)
                if ep < epochs_spec // 2:
                    loss = ce
                else:
                    supcon = supervised_contrastive_loss(feat, y)
                    loss = ce + SUPCON_W_B * supcon
                loss.backward()
                opt2.step()

        sched2.step()

    # -------- Expansion
    new_dim, novelty = compute_novelty(model, spec, test_B, device)
    model = expand_kfn(model, spec, new_dim, novelty, device)

    ema_model = copy.deepcopy(model).to(device)
    ema_model.eval()
    for p in ema_model.parameters():
        p.requires_grad = False

    # -------- Phase 3 (integration)
    for _, p in model.named_parameters():
        p.requires_grad = False

    model.specialists[0].eval()
    model.specialists[1].eval()

    for p in model.global_model.fusion.parameters():
        p.requires_grad = True
    for p in model.global_model.classifier.parameters():
        p.requires_grad = True
    for p in model.global_model.old_gate.parameters():
        p.requires_grad = True
    for p in model.global_model.old_bottleneck.parameters():
        p.requires_grad = True

    opt3 = torch.optim.AdamW([
        {"params": model.global_model.fusion.parameters(), "lr": 2e-3},
        {"params": model.global_model.old_gate.parameters(), "lr": 5e-5},
        {"params": model.global_model.old_bottleneck.parameters(), "lr": 5e-5},
        {"params": model.global_model.classifier.parameters(), "lr": 1e-3},
    ], weight_decay=1e-4)
    sched3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=epochs_B)

    for _ in range(epochs_B):
        model.train()
        model.specialists[0].eval()
        model.specialists[1].eval()

        replay_loader_epoch = make_dynamic_replay_loader(
            train_base, class_to_indices_train, range(OLD_CLASSES), REPLAY_PER_CLASS, BS, NW, rng
        )
        replay_iter = iter(replay_loader_epoch)

        for x_B, y_B in train_B:
            x_B, y_B = x_B.to(device), y_B.to(device)
            try:
                x_A, y_A = next(replay_iter)
            except StopIteration:
                replay_iter = iter(replay_loader_epoch)
                x_A, y_A = next(replay_iter)
            x_A, y_A = x_A.to(device), y_A.to(device)

            opt3.zero_grad()

            if amp_ok:
                with torch.amp.autocast("cuda"):
                    if rng.random() < 0.7:
                        x_mix = torch.cat([x_A, x_B], dim=0)
                        logits_mix, _ = model(x_mix)
                        logits_A = logits_mix[:x_A.size(0)]
                        logits_B = logits_mix[x_A.size(0):]
                    else:
                        logits_A, _ = model(x_A)
                        logits_B, _ = model(x_B)

                    loss_B = F.cross_entropy(logits_B[:, OLD_CLASSES:], y_B - OLD_CLASSES, label_smoothing=0.1)
                    loss_A = F.cross_entropy(logits_A[:, :OLD_CLASSES], y_A)

                    with torch.no_grad():
                        ema_spec = ema_model.specialists[0](x_A)
                        ema_feat = ema_model.global_model.forward_old([ema_spec])
                    new_spec = model.specialists[0](x_A)
                    new_feat = model.global_model.forward_old([new_spec])
                    loss_distill = F.mse_loss(new_feat, ema_feat)

                    with torch.no_grad():
                        ema_logits_A, _ = ema_model(x_A)
                    student_logits_A, _ = model(x_A)

                    T1, T2 = 2.0, 1.0
                    kd_high = F.kl_div(
                        F.log_softmax(student_logits_A[:, :OLD_CLASSES] / T1, dim=1),
                        F.softmax(ema_logits_A[:, :OLD_CLASSES] / T1, dim=1),
                        reduction="batchmean"
                    ) * (T1 * T1)
                    kd_low = F.kl_div(
                        F.log_softmax(student_logits_A[:, :OLD_CLASSES] / T2, dim=1),
                        F.softmax(ema_logits_A[:, :OLD_CLASSES] / T2, dim=1),
                        reduction="batchmean"
                    )
                    loss_kd_logits = 0.7 * kd_high + 0.3 * kd_low

                    loss = (
                        1.5 * loss_B +
                        1.0 * loss_A +
                        0.25 * loss_distill +
                        0.8 * loss_kd_logits
                    )

                scaler.scale(loss).backward()
                scaler.step(opt3)
                scaler.update()
            else:
                if rng.random() < 0.7:
                    x_mix = torch.cat([x_A, x_B], dim=0)
                    logits_mix, _ = model(x_mix)
                    logits_A = logits_mix[:x_A.size(0)]
                    logits_B = logits_mix[x_A.size(0):]
                else:
                    logits_A, _ = model(x_A)
                    logits_B, _ = model(x_B)

                loss_B = F.cross_entropy(logits_B[:, OLD_CLASSES:], y_B - OLD_CLASSES, label_smoothing=0.1)
                loss_A = F.cross_entropy(logits_A[:, :OLD_CLASSES], y_A)

                with torch.no_grad():
                    ema_spec = ema_model.specialists[0](x_A)
                    ema_feat = ema_model.global_model.forward_old([ema_spec])
                new_spec = model.specialists[0](x_A)
                new_feat = model.global_model.forward_old([new_spec])
                loss_distill = F.mse_loss(new_feat, ema_feat)

                with torch.no_grad():
                    ema_logits_A, _ = ema_model(x_A)
                student_logits_A, _ = model(x_A)

                T1, T2 = 2.0, 1.0
                kd_high = F.kl_div(
                    F.log_softmax(student_logits_A[:, :OLD_CLASSES] / T1, dim=1),
                    F.softmax(ema_logits_A[:, :OLD_CLASSES] / T1, dim=1),
                    reduction="batchmean"
                ) * (T1 * T1)
                kd_low = F.kl_div(
                    F.log_softmax(student_logits_A[:, :OLD_CLASSES] / T2, dim=1),
                    F.softmax(ema_logits_A[:, :OLD_CLASSES] / T2, dim=1),
                    reduction="batchmean"
                )
                loss_kd_logits = 0.7 * kd_high + 0.3 * kd_low

                loss = (
                    1.5 * loss_B +
                    1.0 * loss_A +
                    0.25 * loss_distill +
                    0.8 * loss_kd_logits
                )
                loss.backward()
                opt3.step()

            with torch.no_grad():
                w = model.global_model.classifier.weight
                model.global_model.classifier.weight.copy_(F.normalize(w, dim=1))

            update_ema(ema_model, model, EMA_DECAY)

        sched3.step()

    # -------- Phase 4 (bias correction)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.global_model.classifier.parameters():
        p.requires_grad = True

    opt4 = torch.optim.AdamW(model.global_model.classifier.parameters(), lr=5e-5, weight_decay=1e-4)
    sched4 = torch.optim.lr_scheduler.CosineAnnealingLR(opt4, T_max=epochs_bias)

    for _ in range(epochs_bias):
        model.train()
        replay_loader_epoch = make_dynamic_replay_loader(
            train_base, class_to_indices_train, range(OLD_CLASSES), REPLAY_PER_CLASS, BS, NW, rng
        )
        replay_iter = iter(replay_loader_epoch)

        for x_B, y_B in train_B:
            x_B, y_B = x_B.to(device), y_B.to(device)
            try:
                x_A, y_A = next(replay_iter)
            except StopIteration:
                replay_iter = iter(replay_loader_epoch)
                x_A, y_A = next(replay_iter)

            x_A, y_A = x_A.to(device), y_A.to(device)
            x = torch.cat([x_A, x_B], dim=0)
            y = torch.cat([y_A, y_B], dim=0)

            opt4.zero_grad()

            if amp_ok:
                with torch.amp.autocast("cuda"):
                    logits, _ = model(x)
                    loss = F.cross_entropy(logits, y, label_smoothing=0.1)
                scaler.scale(loss).backward()
                scaler.step(opt4)
                scaler.update()
            else:
                logits, _ = model(x)
                loss = F.cross_entropy(logits, y, label_smoothing=0.1)
                loss.backward()
                opt4.step()

        sched4.step()

    acc_A_final_taskaware = evaluate(model, test_A, device, mode="task_A", temp=EVAL_TEMP)
    acc_B_final_taskaware = evaluate(model, test_B, device, mode="task_B", temp=EVAL_TEMP)
    acc_A_final_cil = evaluate(model, test_A, device, mode="cil", temp=EVAL_TEMP)
    acc_B_final_cil = evaluate(model, test_B, device, mode="cil", temp=EVAL_TEMP)

    return {
        "acc_A_init": acc_A_init,
        "acc_A_final_taskaware": acc_A_final_taskaware,
        "acc_B_final_taskaware": acc_B_final_taskaware,
        "acc_A_final_cil": acc_A_final_cil,
        "acc_B_final_cil": acc_B_final_cil,
        "retention_taskaware": (acc_A_final_taskaware / acc_A_init) * 100 if acc_A_init > 0 else 0.0,
        "retention_cil": (acc_A_final_cil / acc_A_init) * 100 if acc_A_init > 0 else 0.0,
        "forgetting_taskaware": acc_A_init - acc_A_final_taskaware,
        "forgetting_cil": acc_A_init - acc_A_final_cil,
        "bwt_taskaware": acc_A_final_taskaware - acc_A_init,
        "bwt_cil": acc_A_final_cil - acc_A_init
    }

# =========================
# RUN
# =========================
def run_all():
    device = get_device(FORCE_CPU)
    print(f"🚀 Environment: {device} | QUICK_RUN={QUICK_RUN} | AMP={USE_AMP and device.type == 'cuda'}")

    # EuroSAT normalization (ImageNet-style works well as baseline)
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)

    t_train = transforms.Compose([
        transforms.RandomResizedCrop(64, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    t_test = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    root = "./data"

    # load once with test transform, then override transform per subset by dataset copy
    base_full = torchvision.datasets.EuroSAT(root=root, download=True, transform=t_test)
    targets = base_full.targets

    # stratified split
    train_idx, test_idx = stratified_train_test_indices(targets, test_ratio=0.2, seed=42)

    # fixed class split for reproducibility
    old_classes = list(range(OLD_CLASSES))      # 0..4
    new_classes = list(range(OLD_CLASSES, TOTAL_CLASSES))  # 5..9

    train_A_idx = filter_indices_by_class(train_idx, targets, old_classes)
    train_B_idx = filter_indices_by_class(train_idx, targets, new_classes)
    test_A_idx = filter_indices_by_class(test_idx, targets, old_classes)
    test_B_idx = filter_indices_by_class(test_idx, targets, new_classes)

    # separate dataset objects for train/test transforms
    train_base = torchvision.datasets.EuroSAT(root=root, download=False, transform=t_train)
    test_base = torchvision.datasets.EuroSAT(root=root, download=False, transform=t_test)

    # class index map for dynamic replay (from train split only)
    class_to_indices_train = build_class_index_map_from_indices(train_base.targets, train_idx)

    loaders = (
        DataLoader(Subset(train_base, train_A_idx), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True),
        DataLoader(Subset(train_base, train_B_idx), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True),
        DataLoader(Subset(test_base, test_A_idx), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),
        DataLoader(Subset(test_base, test_B_idx), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),
    )

    print(f"Dataset sizes -> TrainA: {len(train_A_idx)}, TrainB: {len(train_B_idx)}, TestA: {len(test_A_idx)}, TestB: {len(test_B_idx)}")

    results = []
    meta = {
        "timestamp": str(datetime.utcnow()),
        "dataset": "EuroSAT",
        "quick_run": QUICK_RUN,
        "epochs_A": EPOCHS_A,
        "epochs_B": EPOCHS_B,
        "epochs_bias": EPOCHS_BIAS,
        "replay_per_class": REPLAY_PER_CLASS,
        "eval_temp": EVAL_TEMP,
        "supcon_w_a": SUPCON_W_A,
        "supcon_w_b": SUPCON_W_B,
        "ema_decay": EMA_DECAY,
        "device": str(device),
        "seeds": SEEDS,
        "results": []
    }

    print("\n[1] Running EuroSAT KFN experiments...")
    try:
        for s in SEEDS:
            print(f" -> Seed {s}...")
            r = run_kfn_eurosat(
                s, loaders, EPOCHS_A, EPOCHS_B, EPOCHS_BIAS,
                device, train_base, class_to_indices_train
            )
            results.append(r)
            meta["results"].append({"seed": s, **r})
            save_progress(meta)

            print(
                f"    ↳ Seed {s}: "
                f"A_init(TA)={r['acc_A_init']:.2f} | "
                f"A_final(TA)={r['acc_A_final_taskaware']:.2f} | "
                f"B_final(TA)={r['acc_B_final_taskaware']:.2f} | "
                f"A_final(CIL)={r['acc_A_final_cil']:.2f} | "
                f"B_final(CIL)={r['acc_B_final_cil']:.2f}"
            )

    except KeyboardInterrupt:
        print("\n⛔ Interrupted. Partial results saved to:", SAVE_PATH)
        save_progress(meta)
        return

    if results:
        print("\n" + "=" * 90)
        print("SUMMARY (Mean ± Std)")
        print("=" * 90)

        for k, label in [
            ("acc_A_init", "A Init (Task-aware)"),
            ("acc_A_final_taskaware", "A Final (Task-aware)"),
            ("acc_B_final_taskaware", "B Final (Task-aware)"),
            ("acc_A_final_cil", "A Final (CIL)"),
            ("acc_B_final_cil", "B Final (CIL)")
        ]:
            vals = [r[k] for r in results]
            print(f"{label:26}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

        for k, label in [
            ("retention_taskaware", "Retention (Task-aware)"),
            ("retention_cil", "Retention (CIL)"),
            ("forgetting_taskaware", "Forgetting (Task-aware)"),
            ("forgetting_cil", "Forgetting (CIL)"),
            ("bwt_taskaware", "BWT (Task-aware)"),
            ("bwt_cil", "BWT (CIL)")
        ]:
            vals = [r[k] for r in results]
            print(f"{label:26}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

        print("=" * 90)
        print(f"Saved run log to: {SAVE_PATH}")

if __name__ == "__main__":
    run_all()

🚀 Environment: cuda | QUICK_RUN=True | AMP=True


100%|██████████| 94.3M/94.3M [00:00<00:00, 226MB/s]


Dataset sizes -> TrainA: 11200, TrainB: 10400, TestA: 2800, TestB: 2600

[1] Running EuroSAT KFN experiments...
 -> Seed 0...
    ↳ Seed 0: A_init(TA)=96.68 | A_final(TA)=96.75 | B_final(TA)=96.38 | A_final(CIL)=81.36 | B_final(CIL)=77.35

SUMMARY (Mean ± Std)
A Init (Task-aware)       : 96.68 ± 0.00
A Final (Task-aware)      : 96.75 ± 0.00
B Final (Task-aware)      : 96.38 ± 0.00
A Final (CIL)             : 81.36 ± 0.00
B Final (CIL)             : 77.35 ± 0.00
Retention (Task-aware)    : 100.07 ± 0.00
Retention (CIL)           : 84.15 ± 0.00
Forgetting (Task-aware)   : -0.07 ± 0.00
Forgetting (CIL)          : 15.32 ± 0.00
BWT (Task-aware)          : 0.07 ± 0.00
BWT (CIL)                 : -15.32 ± 0.00
Saved run log to: ./outputs/kfn_eurosat_progress.json
